# Data Splitting Strategy: Train/Test Protocol
Amazon Electronics Reviews — Helpfulness Prediction Pipeline

Author: Sanath | Module: WM9B7 AIDL | April 2026

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

In [ ]:
df = pd.read_parquet('data/processed/s03_filter.parquet')
print(f"Dataset: {df.shape[0]:,} records x {df.shape[1]} columns")
print(f"Timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nRating distribution:\n{df['rating'].value_counts().sort_index()}")

## Why 80/20?

For N=99,627, even 20% test = ~19,900 records. 70/30 wastes ~10K extra on testing. 90/10 risks undersampling rare review types. **80/20 is the consensus for large datasets** (Hastie et al., 2009).

Key considerations:
- **70/30**: Reduces training power for large datasets; wastes test capacity
- **75/25**: Middle ground, but less commonly standardized
- **80/20**: Industry standard; balances training power with test robustness
- **85/15**: Risk of sparse test set for minority classes
- **90/10**: Dangerous for datasets with rare subgroups; test set becomes too small

In [ ]:
splits = {'70/30': 0.30, '75/25': 0.25, '80/20': 0.20, '85/15': 0.15, '90/10': 0.10}
print(f"{'Split':>8} {'Train':>12} {'Test':>12} {'Test per rating★':>20}")
print("-" * 55)
for name, test_pct in splits.items():
    n_test = int(len(df) * test_pct)
    n_train = len(df) - n_test
    per_rating = n_test // 5  # rough estimate
    print(f"{name:>8} {n_train:>12,} {n_test:>12,} {per_rating:>20,}")
print(f"\n✅ Recommended: 80/20 → {int(len(df)*0.8):,} train / {int(len(df)*0.2):,} test")

## Stratification: Preserve Label Distribution

We stratify on `rating` to preserve the 1-5 star distribution in both train and test splits. This ensures:
- Balanced representation of all rating classes
- No evaluation bias from skewed label distributions
- Model robustness across all review sentiment levels

**Critical for imbalanced data**: If 80% of reviews are 5-stars, random split might give train 79% and test 82%. Stratification keeps both at ~80%.

In [ ]:
# Define features (drop text, metadata)
feature_cols = ['review_length', 'rating', 'is_verified', 'image_count', 
                'product_popularity', 'days_since_first_review']
target = 'helpful_vote'

X = df[feature_cols]
y = df[target]

# THE TEAM STANDARD SPLIT — everyone uses this exact config
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['rating']  # preserve rating distribution
)

print(f"Training set: {len(X_train):,} records ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test set:     {len(X_test):,} records ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nRating distribution preserved?")
print(f"{'Rating':>8} {'Full %':>10} {'Train %':>10} {'Test %':>10}")
for r in sorted(df['rating'].unique()):
    full_pct = 100 * (df['rating'] == r).mean()
    train_pct = 100 * (X_train['rating'] == r).mean()
    test_pct = 100 * (X_test['rating'] == r).mean()
    print(f"{r:>8} {full_pct:>10.2f} {train_pct:>10.2f} {test_pct:>10.2f}")
print("\n✅ Distribution preserved by stratification")

## Temporal Analysis: Is There Drift?

Check if review characteristics change significantly over time. If so, temporal split might be more realistic than random split.

**Key metrics to monitor**:
- Review volume trends
- Helpfulness score distribution over time
- Review length evolution
- Verification rate changes

In [ ]:
df['year'] = df['timestamp'].dt.year
yearly = df.groupby('year').agg(
    count=('helpful_vote', 'size'),
    mean_helpful=('helpful_vote', 'mean'),
    median_helpful=('helpful_vote', 'median'),
    mean_length=('review_length', 'mean'),
    verified_pct=('is_verified', 'mean')
).round(3)

print("Yearly statistics:")
print(yearly.to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
yearly['count'].plot(kind='bar', ax=axes[0,0], color='#58A6FF', title='Reviews per Year')
yearly['mean_helpful'].plot(kind='bar', ax=axes[0,1], color='#3FB950', title='Mean Helpful Vote')
yearly['mean_length'].plot(kind='bar', ax=axes[1,0], color='#BC8CFF', title='Mean Review Length')
(yearly['verified_pct']*100).plot(kind='bar', ax=axes[1,1], color='#D29922', title='Verified Purchase %')
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Temporal Analysis: Is There Significant Drift?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Random vs. Temporal Split Comparison

We train models on both random and temporal splits to assess whether temporal drift would invalidate our random split approach.

**Random split**: Models trained/tested on shuffled data (normal ML pipeline)

**Temporal split**: Models trained on historical data, tested on future data (production scenario)

If temporal performance is much worse, we need to acknowledge potential deployment drift.

In [ ]:
# Temporal split: train on everything before last year, test on last year
last_year = df['year'].max()
temporal_train = df[df['year'] < last_year]
temporal_test = df[df['year'] == last_year]

print(f"Temporal split: train on <{last_year} ({len(temporal_train):,}), test on {last_year} ({len(temporal_test):,})")
print(f"Random split:   train ({len(X_train):,}), test ({len(X_test):,})")

# Quick RF model on both
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

# Random split performance
rf.fit(X_train, y_train)
y_pred_random = rf.predict(X_test)
mae_random = mean_absolute_error(y_test, y_pred_random)
r2_random = r2_score(y_test, y_pred_random)

# Temporal split performance
X_temp_train = temporal_train[feature_cols]
y_temp_train = temporal_train[target]
X_temp_test = temporal_test[feature_cols]
y_temp_test = temporal_test[target]

rf2 = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf2.fit(X_temp_train, y_temp_train)
y_pred_temp = rf2.predict(X_temp_test)
mae_temp = mean_absolute_error(y_temp_test, y_pred_temp)
r2_temp = r2_score(y_temp_test, y_pred_temp)

print(f"\n{'Metric':<15} {'Random 80/20':>15} {'Temporal':>15}")
print("-" * 45)
print(f"{'MAE':<15} {mae_random:>15.3f} {mae_temp:>15.3f}")
print(f"{'R²':<15} {r2_random:>15.3f} {r2_temp:>15.3f}")
print(f"\n✅ Random split recommended as primary evaluation")
print(f"📊 Temporal split useful as robustness diagnostic")

## Data Leakage Prevention Checklist

**CRITICAL**: Before any model evaluation, verify you have NOT committed any of these leakage violations:

- [ ] **Never fit preprocessing (scaler, encoder) on combined train+test data**
  - ✅ Fit on X_train only, transform X_test with those parameters

- [ ] **Never use test set information when engineering features**
  - ✅ All feature creation must happen on train set first

- [ ] **Never stratify on target variable directly**
  - ✅ We stratify on `rating` (a feature), not `helpful_vote` (the target)

- [ ] **Never shuffle and split within cross-validation folds**
  - ✅ Use StratifiedKFold with fixed random_state

- [ ] **Never use global statistics (mean, median, std) from full dataset in features**
  - ✅ Compute statistics from training set only

- [ ] **Never perform model selection on test set**
  - ✅ Use cross-validation on training set for hyperparameter tuning

- [ ] **Never pick features based on test set correlations**
  - ✅ Feature selection must occur during training phase only

- [ ] **Never peek at test predictions before final report**
  - ✅ Hold test predictions until after model is finalized

In [ ]:
# 5-Fold Stratified CV within training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Bin helpful_vote for stratification in CV
y_train_binned = pd.qcut(y_train, q=5, labels=False, duplicates='drop')

print("5-Fold Stratified Cross-Validation Protocol:")
print(f"  Training set size: {len(X_train):,}")
print(f"  Fold size: ~{len(X_train)//5:,}")
print(f"  Stratification: helpful_vote quintiles")
print(f"\n  Use this in your code:")
print(f"  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
print(f"\n  Example usage:")
print(f"  scores = cross_val_score(model, X_train, y_train_binned, cv=cv, scoring='r2')")
print(f"  print(f'CV R² scores: {{scores}}')")  
print(f"  print(f'Mean ± Std: {{scores.mean():.3f}} ± {{scores.std():.3f}}')")  

## Team Standard Configuration

### ✅ REQUIRED: Use this exact split for all models

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,           # 80% train / 20% test
    random_state=42,          # reproducibility
    stratify=df['rating']     # preserve rating distribution
)
```

**All team members must use this configuration:**
- Sanath
- Lithika
- Takuya
- Arjun

This ensures:
1. **Reproducibility**: Same random_state = same splits across all notebooks
2. **Comparability**: All models evaluated on identical test sets
3. **Consistency**: Single source of truth for model performance metrics

In [ ]:
# Final verification
print("="*60)
print("SPLIT SUMMARY")
print("="*60)
print(f"Original dataset:  {len(df):>10,} records")
print(f"Training set:      {len(X_train):>10,} records (80.0%)")
print(f"Test set:          {len(X_test):>10,} records (20.0%)")
print(f"\nRandom seed:       {42:>10}")
print(f"Stratification:    {'rating':>10}")
print(f"Feature columns:   {len(feature_cols):>10}")
print(f"Target variable:   {'helpful_vote':>10}")
print("="*60)
print("✅ Ready for model training!")